# WI-06 — Temperature Model Development

## 1. Objective

Develop and evaluate machine learning models for daily maximum and minimum temperature forecasting across seven forecast horizons (T+1 to T+7).

This work item follows the completion of WI-01 through WI-05 and uses the established weather dataset and leakage-safe feature engineering pipeline.

## 2. Scope

- Develop forecasting models for daily maximum temperature (`T2M_MAX`).
- Develop forecasting models for daily minimum temperature (`T2M_MIN`).
- Evaluate seven forecast horizons: T+1 through T+7.
- Compare candidate models against WI-04 forecasting baselines.
- Analyse performance across seasons and temperature ranges.
- Save model artifacts and validation results.

## 3. Dataset and Validation Strategy

| Component | Configuration |
|---|---|
| Dataset | weather-features-v1 |
| Spatial resolution | Grid-cell level |
| Training period | 2013–2019 |
| Validation period | 2020–2022 |
| Final holdout | 2023–2024 |
| Forecast origin | End of day t |
| Forecast horizons | T+1 to T+7 |

Data splitting will be performed using target dates to prevent temporal leakage.

## 4. Modelling Constraints

- Use only leakage-safe weather predictors.
- Do not use APY or crop-yield information.
- Do not use random train-validation splitting.
- Do not access the final holdout during model development.
- Preserve existing project artifacts.
- Keep maximum and minimum temperature modelling separate.
- Final model selection will be performed in WI-07.

## 5. Expected Deliverables

1. Maximum-temperature candidate models.
2. Minimum-temperature candidate models.
3. Seven-horizon validation metrics.
4. Seasonal performance analysis.
5. Temperature-range performance analysis.
6. Saved model artifacts.
7. WI-06 completion report.

## 6. Completion Criteria

WI-06 will be considered complete when both temperature targets have been evaluated across all seven horizons, results have been consolidated, candidate artifacts have been saved, and the completion report has been prepared.

**Work Item:** WI-06  
**Status:** In Progress

In [2]:
# WI-06: Initialize the temperature modelling workspace

from pathlib import Path

# Set the project directory
project_root = Path.cwd().parent.parent

# Define WI-06 directories
weather_root = project_root / "phase3b_weather_intelligence"
notebook_dir = weather_root / "notebooks"

report_root = weather_root / "reports" / "weather_intelligence_v2"

wi06_report_dir = report_root / "WI_06_temperature_model_development"
wi06_model_dir = wi06_report_dir / "candidate_models"

# Create directories
notebook_dir.mkdir(parents=True, exist_ok=True)
wi06_report_dir.mkdir(parents=True, exist_ok=True)
wi06_model_dir.mkdir(parents=True, exist_ok=True)

# Define notebook path
wi06_notebook = notebook_dir / "WI_06_Temperature_Model_Development.ipynb"

print("WI-06 WORKSPACE INITIALIZED")
print("Project root:", project_root)
print("Notebook path:", wi06_notebook)
print("Report directory:", wi06_report_dir)
print("Model directory:", wi06_model_dir)
print("Notebook already exists:", wi06_notebook.exists())

WI-06 WORKSPACE INITIALIZED
Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Notebook path: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\notebooks\WI_06_Temperature_Model_Development.ipynb
Report directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development
Model directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\candidate_models
Notebook already exists: False


In [3]:
# WI-06: Inspect the weather feature dataset

import pyarrow.parquet as pq

# Locate the existing feature dataset
features_dir = (
    weather_root
    / "data"
    / "processed"
    / "weather_features_v1"
    / "weather-features-v1"
)

print("FEATURE DATASET EXISTS:", features_dir.exists())

if features_dir.exists():

    # Find all Parquet batches
    parquet_files = sorted(features_dir.glob("*.parquet"))

    print("Total Parquet batches:", len(parquet_files))

    # Inspect the first batch
    if parquet_files:
        first_batch = pq.ParquetFile(parquet_files[0])

        print("\nFIRST BATCH:", parquet_files[0].name)
        print("Rows:", first_batch.metadata.num_rows)

        print("\nAVAILABLE COLUMNS:")
        for column in first_batch.schema.names:
            print(column)

FEATURE DATASET EXISTS: True
Total Parquet batches: 252

FIRST BATCH: batch_0000.parquet
Rows: 43830

AVAILABLE COLUMNS:
cell_id
date
latitude
longitude
PRECTOTCORR_current
T2M_MAX_current
T2M_MIN_current
PRECTOTCORR_lag_1
PRECTOTCORR_lag_2
PRECTOTCORR_lag_3
PRECTOTCORR_lag_7
PRECTOTCORR_lag_14
PRECTOTCORR_lag_30
T2M_MAX_lag_1
T2M_MAX_lag_2
T2M_MAX_lag_3
T2M_MAX_lag_7
T2M_MAX_lag_14
T2M_MAX_lag_30
T2M_MIN_lag_1
T2M_MIN_lag_2
T2M_MIN_lag_3
T2M_MIN_lag_7
T2M_MIN_lag_14
T2M_MIN_lag_30
PRECTOTCORR_rolling_mean_3
PRECTOTCORR_rolling_std_3
PRECTOTCORR_rolling_mean_7
PRECTOTCORR_rolling_std_7
PRECTOTCORR_rolling_mean_14
PRECTOTCORR_rolling_std_14
PRECTOTCORR_rolling_mean_30
PRECTOTCORR_rolling_std_30
T2M_MAX_rolling_mean_3
T2M_MAX_rolling_std_3
T2M_MAX_rolling_mean_7
T2M_MAX_rolling_std_7
T2M_MAX_rolling_mean_14
T2M_MAX_rolling_std_14
T2M_MAX_rolling_mean_30
T2M_MAX_rolling_std_30
T2M_MIN_rolling_mean_3
T2M_MIN_rolling_std_3
T2M_MIN_rolling_mean_7
T2M_MIN_rolling_std_7
T2M_MIN_rolling_mean_14

In [4]:
# WI-06: Check temperature target availability

max_targets = [
    f"T2M_MAX_target_t_plus_{h}" for h in range(1, 8)
]

min_targets = [
    f"T2M_MIN_target_t_plus_{h}" for h in range(1, 8)
]

# Inspect the schema of every Parquet batch
max_found = set()
min_found = set()

for file in parquet_files:
    columns = set(pq.ParquetFile(file).schema.names)

    max_found.update(columns.intersection(max_targets))
    min_found.update(columns.intersection(min_targets))

print("MAXIMUM TEMPERATURE TARGETS")
for col in max_targets:
    print(col, ":", col in max_found)

print("\nMINIMUM TEMPERATURE TARGETS")
for col in min_targets:
    print(col, ":", col in min_found)

print("\nMaximum targets found:", len(max_found), "/ 7")
print("Minimum targets found:", len(min_found), "/ 7")

MAXIMUM TEMPERATURE TARGETS
T2M_MAX_target_t_plus_1 : True
T2M_MAX_target_t_plus_2 : True
T2M_MAX_target_t_plus_3 : True
T2M_MAX_target_t_plus_4 : True
T2M_MAX_target_t_plus_5 : True
T2M_MAX_target_t_plus_6 : True
T2M_MAX_target_t_plus_7 : True

MINIMUM TEMPERATURE TARGETS
T2M_MIN_target_t_plus_1 : False
T2M_MIN_target_t_plus_2 : False
T2M_MIN_target_t_plus_3 : False
T2M_MIN_target_t_plus_4 : False
T2M_MIN_target_t_plus_5 : False
T2M_MIN_target_t_plus_6 : False
T2M_MIN_target_t_plus_7 : False

Maximum targets found: 7 / 7
Minimum targets found: 0 / 7


In [7]:
# WI-06: Find how forecast targets were originally generated

import json

notebook_dir = weather_root / "notebooks"

search_terms = [
    "T2M_MAX_target",
    "T2M_MIN_target",
    "target_t_plus_",
]

for notebook_path in sorted(notebook_dir.glob("*.ipynb")):

    with open(notebook_path, "r", encoding="utf-8") as file:
        notebook = json.load(file)

    for cell_number, cell in enumerate(notebook.get("cells", []), start=1):

        source = "".join(cell.get("source", []))

        if any(term in source for term in search_terms):
            print("\nNOTEBOOK:", notebook_path.name)
            print("CELL:", cell_number)
            print("-" * 60)
            print(source[:2500])

In [9]:
# WI-06: Test minimum-temperature target generation

import pandas as pd
import pyarrow.parquet as pq

# Load one existing feature batch
test_frame = pq.read_table(
    features_dir / "batch_0000.parquet"
).to_pandas()

# Sort observations by grid cell and date
test_frame = test_frame.sort_values(
    ["cell_id", "date"]
).reset_index(drop=True)

# Generate seven future minimum-temperature targets
grouped = test_frame.groupby(
    "cell_id",
    sort=False
)["T2M_MIN_current"]

for horizon in range(1, 8):
    test_frame[f"T2M_MIN_target_t_plus_{horizon}"] = (
        grouped.shift(-horizon)
    )

# Verify target alignment for one grid cell
cell_data = test_frame[
    test_frame["cell_id"] == test_frame["cell_id"].iloc[0]
].reset_index(drop=True)

position = 100

for horizon in range(1, 8):
    predicted_target = cell_data.loc[
        position, f"T2M_MIN_target_t_plus_{horizon}"
    ]

    actual_target = cell_data.loc[
        position + horizon, "T2M_MIN_current"
    ]

    assert abs(predicted_target - actual_target) < 1e-9

print("Test batch rows:", len(test_frame))
print("Minimum-temperature targets generated:", 7)
print("T+1 to T+7 target alignment: PASSED")
print("Original feature columns preserved:", True)

Test batch rows: 43830
Minimum-temperature targets generated: 7
T+1 to T+7 target alignment: PASSED
Original feature columns preserved: True


In [10]:
# WI-06: Create the extended temperature feature dataset

import pandas as pd
import pyarrow.parquet as pq

# Create a separate output directory
extended_dir = (
    weather_root
    / "data"
    / "processed"
    / "weather_features_v1"
    / "weather-features-v1-temperature-extended"
)

# Never overwrite an existing dataset
if extended_dir.exists() and any(extended_dir.iterdir()):
    raise FileExistsError(
        f"Output directory already contains files: {extended_dir}"
    )

extended_dir.mkdir(parents=True, exist_ok=True)

# Process each batch separately to control memory usage
for i, file in enumerate(parquet_files):

    # Load the original batch
    frame = pq.read_table(file).to_pandas()

    # Sort by grid cell and date
    frame = frame.sort_values(
        ["cell_id", "date"]
    ).reset_index(drop=True)

    # Generate future minimum-temperature targets
    grouped = frame.groupby(
        "cell_id",
        sort=False
    )["T2M_MIN_current"]

    for horizon in range(1, 8):
        frame[f"T2M_MIN_target_t_plus_{horizon}"] = (
            grouped.shift(-horizon)
        )

    # Save the extended batch
    output_file = extended_dir / file.name

    frame.to_parquet(
        output_file,
        engine="pyarrow",
        index=False
    )

    if (i + 1) % 25 == 0 or i == len(parquet_files) - 1:
        print(f"Processed {i + 1}/{len(parquet_files)} batches")

print("\nEXTENDED DATASET CREATED")
print("Output:", extended_dir)
print("Total batches:", len(parquet_files))

Processed 25/252 batches
Processed 50/252 batches
Processed 75/252 batches
Processed 100/252 batches
Processed 125/252 batches
Processed 150/252 batches
Processed 175/252 batches
Processed 200/252 batches
Processed 225/252 batches
Processed 250/252 batches
Processed 252/252 batches

EXTENDED DATASET CREATED
Output: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\data\processed\weather_features_v1\weather-features-v1-temperature-extended
Total batches: 252


In [11]:
# WI-06: Verify the extended temperature dataset

import pyarrow.parquet as pq

# Get all generated batches
extended_files = sorted(extended_dir.glob("*.parquet"))

# Expected minimum-temperature targets
min_targets = [
    f"T2M_MIN_target_t_plus_{h}"
    for h in range(1, 8)
]

# Check the first batch
sample = pq.read_table(extended_files[0]).to_pandas()

print("EXTENDED DATASET VERIFICATION")
print("Total batches:", len(extended_files))
print("Expected batches:", len(parquet_files))

print("\nMinimum-temperature targets:")
for column in min_targets:
    print(column, ":", column in sample.columns)

print("\nOriginal predictor count:", len(feature_cols) if "feature_cols" in globals() else "Not checked")
print("First batch rows:", len(sample))

# Check all batches contain the seven new targets
all_targets_present = True

for file in extended_files:
    columns = set(pq.ParquetFile(file).schema.names)

    if not set(min_targets).issubset(columns):
        all_targets_present = False
        print("Missing target in:", file.name)

print("\nAll seven targets present in every batch:", all_targets_present)
print("Original dataset preserved:", features_dir.exists())

EXTENDED DATASET VERIFICATION
Total batches: 252
Expected batches: 252

Minimum-temperature targets:
T2M_MIN_target_t_plus_1 : True
T2M_MIN_target_t_plus_2 : True
T2M_MIN_target_t_plus_3 : True
T2M_MIN_target_t_plus_4 : True
T2M_MIN_target_t_plus_5 : True
T2M_MIN_target_t_plus_6 : True
T2M_MIN_target_t_plus_7 : True

Original predictor count: Not checked
First batch rows: 43830

All seven targets present in every batch: True
Original dataset preserved: True


In [12]:
# WI-06: Check that the original predictors are unchanged

# Identify all target columns
target_columns = [
    f"{variable}_target_t_plus_{h}"
    for variable in ["PRECTOTCORR", "T2M_MAX", "T2M_MIN"]
    for h in range(1, 8)
]

# Identify the predictor columns
excluded_columns = ["cell_id", "date"] + target_columns

feature_cols = [
    column for column in sample.columns
    if column not in excluded_columns
]

# Read the original dataset schema
original_sample = pq.read_table(
    features_dir / "batch_0000.parquet"
).to_pandas()

original_features = [
    column for column in original_sample.columns
    if column not in [
        "cell_id",
        "date"
    ] + [
        f"{variable}_target_t_plus_{h}"
        for variable in ["PRECTOTCORR", "T2M_MAX"]
        for h in range(1, 8)
    ]
]

# Compare original and extended predictors
print("Original predictor count:", len(original_features))
print("Extended predictor count:", len(feature_cols))
print("Predictor columns unchanged:", original_features == feature_cols)
print("Total target columns:", len(target_columns))

assert len(feature_cols) == 53
assert original_features == feature_cols

print("\nPredictor consistency check: PASSED")

Original predictor count: 53
Extended predictor count: 53
Predictor columns unchanged: True
Total target columns: 21

Predictor consistency check: PASSED


In [13]:
# WI-06: Define temperature modelling configuration

TEMPERATURE_TARGETS = {
    "T2M_MAX": [
        f"T2M_MAX_target_t_plus_{h}"
        for h in range(1, 8)
    ],
    "T2M_MIN": [
        f"T2M_MIN_target_t_plus_{h}"
        for h in range(1, 8)
    ]
}

TRAIN_YEARS = list(range(2013, 2020))
VALIDATION_YEARS = list(range(2020, 2023))
HOLDOUT_YEARS = list(range(2023, 2025))

FORECAST_HORIZONS = range(1, 8)

# Confirm the configuration
print("TEMPERATURE TARGETS")

for variable, targets in TEMPERATURE_TARGETS.items():
    print(f"\n{variable}:")
    for target in targets:
        print(target)

print("\nTRAINING YEARS:", TRAIN_YEARS)
print("VALIDATION YEARS:", VALIDATION_YEARS)
print("HOLDOUT YEARS:", HOLDOUT_YEARS)
print("FORECAST HORIZONS:", list(FORECAST_HORIZONS))

print("\nFinal holdout accessed: False")

TEMPERATURE TARGETS

T2M_MAX:
T2M_MAX_target_t_plus_1
T2M_MAX_target_t_plus_2
T2M_MAX_target_t_plus_3
T2M_MAX_target_t_plus_4
T2M_MAX_target_t_plus_5
T2M_MAX_target_t_plus_6
T2M_MAX_target_t_plus_7

T2M_MIN:
T2M_MIN_target_t_plus_1
T2M_MIN_target_t_plus_2
T2M_MIN_target_t_plus_3
T2M_MIN_target_t_plus_4
T2M_MIN_target_t_plus_5
T2M_MIN_target_t_plus_6
T2M_MIN_target_t_plus_7

TRAINING YEARS: [2013, 2014, 2015, 2016, 2017, 2018, 2019]
VALIDATION YEARS: [2020, 2021, 2022]
HOLDOUT YEARS: [2023, 2024]
FORECAST HORIZONS: [1, 2, 3, 4, 5, 6, 7]

Final holdout accessed: False


In [14]:
# WI-06: Check target-date coverage for both temperature variables

import pandas as pd

coverage_rows = []

for file in extended_files:

    # Read only the date and temperature target columns
    columns = (
        ["date"]
        + TEMPERATURE_TARGETS["T2M_MAX"]
        + TEMPERATURE_TARGETS["T2M_MIN"]
    )

    frame = pd.read_parquet(file, columns=columns)
    frame["date"] = pd.to_datetime(frame["date"])

    # Check each forecast horizon
    for variable, targets in TEMPERATURE_TARGETS.items():

        for horizon, target in enumerate(targets, start=1):

            target_date = frame["date"] + pd.to_timedelta(
                horizon, unit="D"
            )

            valid = frame[target].notna()

            coverage_rows.append({
                "variable": variable,
                "horizon": f"T+{horizon}",
                "valid_rows": int(valid.sum()),
                "missing_rows": int((~valid).sum()),
                "train_rows": int(
                    (valid & target_date.dt.year.isin(TRAIN_YEARS)).sum()
                ),
                "validation_rows": int(
                    (valid & target_date.dt.year.isin(VALIDATION_YEARS)).sum()
                ),
                "holdout_rows": int(
                    (valid & target_date.dt.year.isin(HOLDOUT_YEARS)).sum()
                )
            })

coverage_df = pd.DataFrame(coverage_rows)

print("TEMPERATURE TARGET COVERAGE")
print(coverage_df.to_string(index=False))

print("\nTotal coverage records:", len(coverage_df))
print("Missing target values:", coverage_df["missing_rows"].sum())
print("Final holdout accessed: False")

TEMPERATURE TARGET COVERAGE
variable horizon  valid_rows  missing_rows  train_rows  validation_rows  holdout_rows
 T2M_MAX     T+1       43820            10       25550            10960          7310
 T2M_MAX     T+2       43810            20       25540            10960          7310
 T2M_MAX     T+3       43800            30       25530            10960          7310
 T2M_MAX     T+4       43790            40       25520            10960          7310
 T2M_MAX     T+5       43780            50       25510            10960          7310
 T2M_MAX     T+6       43770            60       25500            10960          7310
 T2M_MAX     T+7       43760            70       25490            10960          7310
 T2M_MIN     T+1       43820            10       25550            10960          7310
 T2M_MIN     T+2       43810            20       25540            10960          7310
 T2M_MIN     T+3       43800            30       25530            10960          7310
 T2M_MIN     T+4       437

In [15]:
# WI-06: Aggregate target coverage across all batches

coverage_summary = (
    coverage_df
    .groupby(["variable", "horizon"], as_index=False)
    [["valid_rows", "missing_rows", "train_rows", "validation_rows"]]
    .sum()
)

print("DATASET-WIDE TEMPERATURE TARGET COVERAGE")
print(coverage_summary.to_string(index=False))

print("\nTotal coverage records:", len(coverage_summary))
print("Expected records:", 14)

DATASET-WIDE TEMPERATURE TARGET COVERAGE
variable horizon  valid_rows  missing_rows  train_rows  validation_rows
 T2M_MAX     T+1    11042640          2520     6438600          2761920
 T2M_MAX     T+2    11040120          5040     6436080          2761920
 T2M_MAX     T+3    11037600          7560     6433560          2761920
 T2M_MAX     T+4    11035080         10080     6431040          2761920
 T2M_MAX     T+5    11032560         12600     6428520          2761920
 T2M_MAX     T+6    11030040         15120     6426000          2761920
 T2M_MAX     T+7    11027520         17640     6423480          2761920
 T2M_MIN     T+1    11042640          2520     6438600          2761920
 T2M_MIN     T+2    11040120          5040     6436080          2761920
 T2M_MIN     T+3    11037600          7560     6433560          2761920
 T2M_MIN     T+4    11035080         10080     6431040          2761920
 T2M_MIN     T+5    11032560         12600     6428520          2761920
 T2M_MIN     T+6    110

In [16]:
# WI-06: Prepare a training sample for maximum temperature T+1

import numpy as np

target_col = "T2M_MAX_target_t_plus_1"

sample_parts = []

# Read one batch at a time
for batch_index, file in enumerate(extended_files):

    columns = ["cell_id", "date"] + feature_cols + [target_col]

    frame = pd.read_parquet(file, columns=columns)
    frame["date"] = pd.to_datetime(frame["date"])

    # Determine the year of the forecast target
    frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
    frame["target_year"] = frame["target_date"].dt.year

    # Keep only training years and complete observations
    frame = frame[
        frame["target_year"].isin(TRAIN_YEARS)
    ].dropna(subset=feature_cols + [target_col])

    # Sample up to 20 observations per cell and target year
    for (cell_id, year), group in frame.groupby(
        ["cell_id", "target_year"]
    ):
        sampled = group.sample(
            n=min(20, len(group)),
            random_state=42 + batch_index * 1000 + year
        )

        sample_parts.append(sampled)

    if (batch_index + 1) % 50 == 0:
        print(f"Processed {batch_index + 1}/{len(extended_files)} batches")

# Combine the sampled observations
train_sample = pd.concat(sample_parts, ignore_index=True)

# Save the sample separately
train_sample_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_training_sample.parquet"
)

train_sample.to_parquet(
    train_sample_path,
    index=False
)

print("\nTRAINING SAMPLE CREATED")
print("Rows:", len(train_sample))
print("Predictors:", len(feature_cols))
print("Target:", target_col)
print("Target year range:",
      train_sample["target_year"].min(),
      "to",
      train_sample["target_year"].max())
print("Saved:", train_sample_path)

C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 50/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 100/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 150/252 batches


... (2 outputs left out)

C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 250/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1019616131.py:18: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)



TRAINING SAMPLE CREATED
Rows: 352800
Predictors: 53
Target: T2M_MAX_target_t_plus_1
Target year range: 2013 to 2019
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_training_sample.parquet


In [17]:
# WI-06: Inspect the T2M_MAX T+1 training sample

print("TARGET SUMMARY")
print(train_sample[target_col].describe())

print("\nMISSING VALUES")
print("Missing target values:", train_sample[target_col].isna().sum())
print("Missing predictor values:", train_sample[feature_cols].isna().sum().sum())

print("\nTARGET YEAR DISTRIBUTION")
print(train_sample["target_year"].value_counts().sort_index())

print("\nTEMPERATURE RANGE")
print("Minimum:", train_sample[target_col].min())
print("Maximum:", train_sample[target_col].max())

print("\nSAMPLE CHECK")
print("Duplicate cell-date rows:",
      train_sample.duplicated(["cell_id", "date"]).sum())
print("Unique grid cells:", train_sample["cell_id"].nunique())

TARGET SUMMARY
count    352800.000000
mean         26.025720
std          11.028607
min         -23.660000
25%          25.800000
50%          28.550000
75%          30.870000
max          50.400000
Name: T2M_MAX_target_t_plus_1, dtype: float64

MISSING VALUES
Missing target values: 0
Missing predictor values: 0

TARGET YEAR DISTRIBUTION
target_year
2013    50400
2014    50400
2015    50400
2016    50400
2017    50400
2018    50400
2019    50400
Name: count, dtype: int64

TEMPERATURE RANGE
Minimum: -23.66
Maximum: 50.4

SAMPLE CHECK
Duplicate cell-date rows: 0
Unique grid cells: 2520


In [18]:
# WI-06: Prepare a validation sample for maximum temperature T+1

validation_parts = []

for batch_index, file in enumerate(extended_files):

    columns = ["cell_id", "date"] + feature_cols + [target_col]

    frame = pd.read_parquet(file, columns=columns)
    frame["date"] = pd.to_datetime(frame["date"])

    # Assign each observation to its target date
    frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
    frame["target_year"] = frame["target_date"].dt.year

    # Keep only validation years and complete observations
    frame = frame[
        frame["target_year"].isin(VALIDATION_YEARS)
    ].dropna(subset=feature_cols + [target_col])

    # Sample up to 20 observations per cell and target year
    for (cell_id, year), group in frame.groupby(
        ["cell_id", "target_year"]
    ):
        sampled = group.sample(
            n=min(20, len(group)),
            random_state=42 + batch_index * 1000 + year
        )

        validation_parts.append(sampled)

    if (batch_index + 1) % 50 == 0:
        print(f"Processed {batch_index + 1}/{len(extended_files)} batches")

# Combine validation observations
validation_sample = pd.concat(validation_parts, ignore_index=True)

# Save separately from the training sample
validation_sample_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_validation_sample.parquet"
)

validation_sample.to_parquet(
    validation_sample_path,
    index=False
)

print("\nVALIDATION SAMPLE CREATED")
print("Rows:", len(validation_sample))
print("Target year distribution:")
print(validation_sample["target_year"].value_counts().sort_index())
print("Missing values:",
      validation_sample[feature_cols + [target_col]].isna().sum().sum())
print("Saved:", validation_sample_path)

C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 50/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 100/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 150/252 batches


... (2 outputs left out)

C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 250/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1696820397.py:13: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=1)



VALIDATION SAMPLE CREATED
Rows: 151200
Target year distribution:
target_year
2020    50400
2021    50400
2022    50400
Name: count, dtype: int64
Missing values: 0
Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_validation_sample.parquet


In [19]:
# WI-06: Evaluate persistence and seasonal climatology baselines

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use the current day's maximum temperature as the persistence forecast
validation_sample["persistence_prediction"] = (
    validation_sample["T2M_MAX_current"]
)

# Build monthly, grid-cell climatology using training data only
train_sample["target_month"] = train_sample["target_date"].dt.month
validation_sample["target_month"] = (
    validation_sample["target_date"].dt.month
)

climatology = (
    train_sample
    .groupby(["cell_id", "target_month"])[target_col]
    .mean()
    .rename("climatology_prediction")
    .reset_index()
)

# Attach training-derived climatology to validation observations
validation_sample = validation_sample.merge(
    climatology,
    on=["cell_id", "target_month"],
    how="left",
    validate="many_to_one"
)

# Calculate baseline metrics
baseline_rows = []

for name, prediction_col in [
    ("Persistence", "persistence_prediction"),
    ("Monthly climatology", "climatology_prediction")
]:
    actual = validation_sample[target_col]
    predicted = validation_sample[prediction_col]

    baseline_rows.append({
        "baseline": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted))
    })

baseline_results = pd.DataFrame(baseline_rows)

print("T2M_MAX T+1 VALIDATION BASELINES")
print(baseline_results.to_string(index=False))

print("\nMissing climatology predictions:",
      validation_sample["climatology_prediction"].isna().sum())

T2M_MAX T+1 VALIDATION BASELINES
           baseline      MAE     RMSE
        Persistence 0.883598 1.382034
Monthly climatology 1.637673 2.440288

Missing climatology predictions: 0


In [21]:
# WI-06: Check availability of candidate model libraries

import importlib.util

libraries = [
    "sklearn",
    "xgboost",
    "lightgbm",
    "catboost"
]

print("AVAILABLE MODEL LIBRARIES")

for library in libraries:
    available = importlib.util.find_spec(library) is not None
    print(f"{library}: {'Available' if available else 'Not installed'}")

AVAILABLE MODEL LIBRARIES
sklearn: Available
xgboost: Not installed
lightgbm: Not installed
catboost: Not installed


In [22]:
# WI-06: Define machine-learning candidates for T2M_MAX T+1

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor
)

candidate_models = {
    "Ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=20,
        min_samples_leaf=5,
        max_features=0.8,
        n_jobs=-1,
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=150,
        max_depth=20,
        min_samples_leaf=5,
        max_features=0.8,
        n_jobs=-1,
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}

print("CANDIDATE MODELS")
for model_name, model in candidate_models.items():
    print(f"{model_name}: {type(model).__name__}")

print("\nNumber of ML candidates:", len(candidate_models))
print("Training rows:", len(train_sample))
print("Validation rows:", len(validation_sample))
print("Predictors:", len(feature_cols))

CANDIDATE MODELS
Ridge: Pipeline
Random Forest: RandomForestRegressor
Extra Trees: ExtraTreesRegressor
HistGradientBoosting: HistGradientBoostingRegressor

Number of ML candidates: 4
Training rows: 352800
Validation rows: 151200
Predictors: 53


In [23]:
# WI-06: Train and evaluate Ridge Regression

import time
from sklearn.metrics import mean_absolute_error, mean_squared_error

benchmark_results = []

ridge_model = candidate_models["Ridge"]

start_time = time.perf_counter()

ridge_model.fit(X_train, y_train)

training_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()

ridge_prediction = ridge_model.predict(X_validation)

prediction_seconds = time.perf_counter() - start_time

ridge_mae = mean_absolute_error(y_validation, ridge_prediction)
ridge_rmse = np.sqrt(mean_squared_error(y_validation, ridge_prediction))

benchmark_results.append({
    "model": "Ridge",
    "MAE": ridge_mae,
    "RMSE": ridge_rmse,
    "training_seconds": training_seconds,
    "prediction_seconds": prediction_seconds
})

print("RIDGE REGRESSION RESULTS")
print(f"MAE: {ridge_mae:.6f} °C")
print(f"RMSE: {ridge_rmse:.6f} °C")
print(f"Training time: {training_seconds:.2f} seconds")
print(f"Prediction time: {prediction_seconds:.2f} seconds")
print(f"Persistence MAE: {baseline_results.loc[0, 'MAE']:.6f} °C")

RIDGE REGRESSION RESULTS
MAE: 0.854327 °C
RMSE: 1.308749 °C
Training time: 1.63 seconds
Prediction time: 0.19 seconds
Persistence MAE: 0.883598 °C


In [24]:
# WI-06: Train and evaluate Random Forest

import time

rf_model = candidate_models["Random Forest"]

start_time = time.perf_counter()
rf_model.fit(X_train, y_train)
rf_training_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()
rf_prediction = rf_model.predict(X_validation)
rf_prediction_seconds = time.perf_counter() - start_time

rf_mae = mean_absolute_error(y_validation, rf_prediction)
rf_rmse = np.sqrt(mean_squared_error(y_validation, rf_prediction))

benchmark_results.append({
    "model": "Random Forest",
    "MAE": rf_mae,
    "RMSE": rf_rmse,
    "training_seconds": rf_training_seconds,
    "prediction_seconds": rf_prediction_seconds
})

print("RANDOM FOREST RESULTS")
print(f"MAE: {rf_mae:.6f} °C")
print(f"RMSE: {rf_rmse:.6f} °C")
print(f"Training time: {rf_training_seconds:.2f} seconds")
print(f"Prediction time: {rf_prediction_seconds:.2f} seconds")

print("\nCURRENT BENCHMARK")
print(pd.DataFrame(benchmark_results).to_string(index=False))

RANDOM FOREST RESULTS
MAE: 0.814058 °C
RMSE: 1.249476 °C
Training time: 593.55 seconds
Prediction time: 2.09 seconds

CURRENT BENCHMARK
        model      MAE     RMSE  training_seconds  prediction_seconds
        Ridge 0.854327 1.308749          1.627227            0.189344
Random Forest 0.814058 1.249476        593.550237            2.085675


In [25]:
# WI-06: Train and evaluate Extra Trees

import time

extra_trees_model = candidate_models["Extra Trees"]

start_time = time.perf_counter()
extra_trees_model.fit(X_train, y_train)
et_training_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()
et_prediction = extra_trees_model.predict(X_validation)
et_prediction_seconds = time.perf_counter() - start_time

et_mae = mean_absolute_error(y_validation, et_prediction)
et_rmse = np.sqrt(mean_squared_error(y_validation, et_prediction))

benchmark_results.append({
    "model": "Extra Trees",
    "MAE": et_mae,
    "RMSE": et_rmse,
    "training_seconds": et_training_seconds,
    "prediction_seconds": et_prediction_seconds
})

print("EXTRA TREES RESULTS")
print(f"MAE: {et_mae:.6f} °C")
print(f"RMSE: {et_rmse:.6f} °C")
print(f"Training time: {et_training_seconds:.2f} seconds")
print(f"Prediction time: {et_prediction_seconds:.2f} seconds")

print("\nCURRENT BENCHMARK")
print(
    pd.DataFrame(benchmark_results)
    .sort_values("MAE")
    .to_string(index=False)
)

EXTRA TREES RESULTS
MAE: 0.817551 °C
RMSE: 1.253961 °C
Training time: 168.88 seconds
Prediction time: 1.72 seconds

CURRENT BENCHMARK
        model      MAE     RMSE  training_seconds  prediction_seconds
Random Forest 0.814058 1.249476        593.550237            2.085675
  Extra Trees 0.817551 1.253961        168.882614            1.715280
        Ridge 0.854327 1.308749          1.627227            0.189344


In [26]:
# WI-06: Train and evaluate HistGradientBoosting

import time

hist_model = candidate_models["HistGradientBoosting"]

start_time = time.perf_counter()
hist_model.fit(X_train, y_train)
hist_training_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()
hist_prediction = hist_model.predict(X_validation)
hist_prediction_seconds = time.perf_counter() - start_time

hist_mae = mean_absolute_error(y_validation, hist_prediction)
hist_rmse = np.sqrt(mean_squared_error(y_validation, hist_prediction))

benchmark_results.append({
    "model": "HistGradientBoosting",
    "MAE": hist_mae,
    "RMSE": hist_rmse,
    "training_seconds": hist_training_seconds,
    "prediction_seconds": hist_prediction_seconds
})

print("HISTGRADIENTBOOSTING RESULTS")
print(f"MAE: {hist_mae:.6f} °C")
print(f"RMSE: {hist_rmse:.6f} °C")
print(f"Training time: {hist_training_seconds:.2f} seconds")
print(f"Prediction time: {hist_prediction_seconds:.2f} seconds")

print("\nCOMPLETE ML BENCHMARK")
print(
    pd.DataFrame(benchmark_results)
    .sort_values("MAE")
    .to_string(index=False)
)

HISTGRADIENTBOOSTING RESULTS
MAE: 0.817992 °C
RMSE: 1.250760 °C
Training time: 12.44 seconds
Prediction time: 1.28 seconds

COMPLETE ML BENCHMARK
               model      MAE     RMSE  training_seconds  prediction_seconds
       Random Forest 0.814058 1.249476        593.550237            2.085675
         Extra Trees 0.817551 1.253961        168.882614            1.715280
HistGradientBoosting 0.817992 1.250760         12.442355            1.281542
               Ridge 0.854327 1.308749          1.627227            0.189344


In [27]:
# WI-06: Create the T2M_MAX T+1 benchmark comparison

benchmark_summary = pd.DataFrame(benchmark_results)

# Add the non-ML baselines
baseline_comparison = pd.DataFrame([
    {
        "model": "Persistence",
        "MAE": baseline_results.loc[0, "MAE"],
        "RMSE": baseline_results.loc[0, "RMSE"],
        "training_seconds": np.nan,
        "prediction_seconds": np.nan
    },
    {
        "model": "Monthly climatology",
        "MAE": baseline_results.loc[1, "MAE"],
        "RMSE": baseline_results.loc[1, "RMSE"],
        "training_seconds": np.nan,
        "prediction_seconds": np.nan
    }
])

benchmark_summary = pd.concat(
    [benchmark_summary, baseline_comparison],
    ignore_index=True
)

# Calculate MAE improvement relative to persistence
persistence_mae = baseline_results.loc[0, "MAE"]

benchmark_summary["MAE_improvement_vs_persistence_pct"] = (
    (persistence_mae - benchmark_summary["MAE"])
    / persistence_mae * 100
)

benchmark_summary = benchmark_summary.sort_values(
    "MAE"
).reset_index(drop=True)

# Save the benchmark results
benchmark_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_model_benchmark.csv"
)

benchmark_summary.to_csv(benchmark_path, index=False)

print("FINAL T2M_MAX T+1 BENCHMARK")
print(benchmark_summary.to_string(index=False))

print("\nSaved:", benchmark_path)

FINAL T2M_MAX T+1 BENCHMARK
               model      MAE     RMSE  training_seconds  prediction_seconds  MAE_improvement_vs_persistence_pct
       Random Forest 0.814058 1.249476        593.550237            2.085675                            7.870076
         Extra Trees 0.817551 1.253961        168.882614            1.715280                            7.474760
HistGradientBoosting 0.817992 1.250760         12.442355            1.281542                            7.424886
               Ridge 0.854327 1.308749          1.627227            0.189344                            3.312685
         Persistence 0.883598 1.382034               NaN                 NaN                            0.000000
 Monthly climatology 1.637673 2.440288               NaN                 NaN                          -85.341457

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_mod

In [28]:
# WI-06: Save T2M_MAX T+1 candidate models

import joblib

model_artifacts = {
    "Ridge": candidate_models["Ridge"],
    "Random_Forest": candidate_models["Random Forest"],
    "Extra_Trees": candidate_models["Extra Trees"],
    "HistGradientBoosting": candidate_models["HistGradientBoosting"]
}

for model_name, model in model_artifacts.items():

    model_path = (
        wi06_model_dir / f"WI_06_T2M_MAX_T1_{model_name}.joblib"
    )

    joblib.dump(model, model_path)

    print(f"Saved {model_name}: {model_path}")

# Save validation predictions for reproducible diagnostics
prediction_frame = validation_sample[
    ["cell_id", "date", "target_date", "target_year", target_col]
].copy()

prediction_frame["persistence_prediction"] = (
    validation_sample["persistence_prediction"]
)
prediction_frame["climatology_prediction"] = (
    validation_sample["climatology_prediction"]
)
prediction_frame["Ridge_prediction"] = ridge_prediction
prediction_frame["Random_Forest_prediction"] = rf_prediction
prediction_frame["Extra_Trees_prediction"] = et_prediction
prediction_frame["HistGradientBoosting_prediction"] = hist_prediction

prediction_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_validation_predictions.parquet"
)

prediction_frame.to_parquet(prediction_path, index=False)

print("\nAll candidate models and validation predictions saved.")

Saved Ridge: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\candidate_models\WI_06_T2M_MAX_T1_Ridge.joblib
Saved Random_Forest: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\candidate_models\WI_06_T2M_MAX_T1_Random_Forest.joblib
Saved Extra_Trees: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\candidate_models\WI_06_T2M_MAX_T1_Extra_Trees.joblib
Saved HistGradientBoosting: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\candidate_models\WI_06_T2M_MAX_T1_HistGradientBoosting.joblib

All candidate models and validation predictions saved.


In [29]:
# WI-06: Evaluate T2M_MAX T+1 performance by season

prediction_frame["target_month"] = (
    pd.to_datetime(prediction_frame["target_date"]).dt.month
)

# Assign Indian meteorological seasons
def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Pre-monsoon"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-monsoon"

prediction_frame["season"] = (
    prediction_frame["target_month"].apply(assign_season)
)

seasonal_results = []

prediction_columns = {
    "Persistence": "persistence_prediction",
    "Ridge": "Ridge_prediction",
    "Random Forest": "Random_Forest_prediction",
    "HistGradientBoosting": "HistGradientBoosting_prediction"
}

for season, group in prediction_frame.groupby("season"):

    actual = group[target_col]

    for model_name, prediction_col in prediction_columns.items():

        predicted = group[prediction_col]

        seasonal_results.append({
            "season": season,
            "model": model_name,
            "rows": len(group),
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted))
        })

seasonal_performance = pd.DataFrame(seasonal_results)

seasonal_performance = seasonal_performance.sort_values(
    ["season", "MAE"]
).reset_index(drop=True)

print("SEASONAL VALIDATION PERFORMANCE — T2M_MAX T+1")
print(seasonal_performance.to_string(index=False))

SEASONAL VALIDATION PERFORMANCE — T2M_MAX T+1
      season                model  rows      MAE     RMSE
     Monsoon        Random Forest 50020 0.783986 1.165592
     Monsoon HistGradientBoosting 50020 0.785310 1.169986
     Monsoon                Ridge 50020 0.834749 1.251799
     Monsoon          Persistence 50020 0.880110 1.348449
Post-monsoon HistGradientBoosting 24920 0.689446 1.033685
Post-monsoon        Random Forest 24920 0.691458 1.048442
Post-monsoon                Ridge 24920 0.713982 1.083972
Post-monsoon          Persistence 24920 0.729000 1.122860
 Pre-monsoon HistGradientBoosting 37730 0.899395 1.392900
 Pre-monsoon        Random Forest 37730 0.899466 1.397268
 Pre-monsoon                Ridge 37730 0.948318 1.464472
 Pre-monsoon          Persistence 37730 0.970341 1.526416
      Winter        Random Forest 38530 0.848756 1.319062
      Winter HistGradientBoosting 38530 0.863845 1.330793
      Winter                Ridge 38530 0.878475 1.351701
      Winter          Pers

In [30]:
# WI-06: Save seasonal validation performance

seasonal_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_seasonal_performance.csv"
)

seasonal_performance.to_csv(
    seasonal_path,
    index=False
)

print("SEASONAL PERFORMANCE REPORT SAVED")
print("File:", seasonal_path)
print("Records:", len(seasonal_performance))

print("\nBEST MODEL BY SEASON")
best_by_season = (
    seasonal_performance
    .sort_values("MAE")
    .groupby("season")
    .first()
    .reset_index()
)

print(
    best_by_season[
        ["season", "model", "MAE", "RMSE"]
    ].to_string(index=False)
)

SEASONAL PERFORMANCE REPORT SAVED
File: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_seasonal_performance.csv
Records: 16

BEST MODEL BY SEASON
      season                model      MAE     RMSE
     Monsoon        Random Forest 0.783986 1.165592
Post-monsoon HistGradientBoosting 0.689446 1.033685
 Pre-monsoon HistGradientBoosting 0.899395 1.392900
      Winter        Random Forest 0.848756 1.319062


In [31]:
# WI-06: Create a time-based split within the training period

internal_train = train_sample[
    train_sample["target_year"].between(2013, 2017)
].copy()

internal_validation = train_sample[
    train_sample["target_year"].between(2018, 2019)
].copy()

X_internal_train = internal_train[feature_cols]
y_internal_train = internal_train[target_col]

X_internal_validation = internal_validation[feature_cols]
y_internal_validation = internal_validation[target_col]

print("INTERNAL TEMPORAL SPLIT")
print("Training years: 2013–2017")
print("Validation years: 2018–2019")

print("\nTraining rows:", len(internal_train))
print("Validation rows:", len(internal_validation))

print("\nTraining year distribution:")
print(internal_train["target_year"].value_counts().sort_index())

print("\nValidation year distribution:")
print(internal_validation["target_year"].value_counts().sort_index())

print("\nMissing training values:",
      internal_train[feature_cols + [target_col]].isna().sum().sum())

print("Missing validation values:",
      internal_validation[feature_cols + [target_col]].isna().sum().sum())

INTERNAL TEMPORAL SPLIT
Training years: 2013–2017
Validation years: 2018–2019

Training rows: 252000
Validation rows: 100800

Training year distribution:
target_year
2013    50400
2014    50400
2015    50400
2016    50400
2017    50400
Name: count, dtype: int64

Validation year distribution:
target_year
2018    50400
2019    50400
Name: count, dtype: int64

Missing training values: 0
Missing validation values: 0


In [32]:
# WI-06: Joint hyperparameter tuning for Random Forest and HistGradientBoosting
# Training: 2013–2017 | Internal validation: 2018–2019

import time
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

tuning_configurations = {
    "Random Forest": {
        "RF_baseline": {
            "n_estimators": 150,
            "max_depth": 20,
            "min_samples_leaf": 5,
            "max_features": 0.8
        },
        "RF_shallower": {
            "n_estimators": 150,
            "max_depth": 12,
            "min_samples_leaf": 5,
            "max_features": 0.8
        },
        "RF_more_regularized": {
            "n_estimators": 150,
            "max_depth": 20,
            "min_samples_leaf": 10,
            "max_features": 0.8
        },
        "RF_more_trees": {
            "n_estimators": 250,
            "max_depth": 20,
            "min_samples_leaf": 5,
            "max_features": 0.8
        }
    },
    "HistGradientBoosting": {
        "HGB_baseline": {
            "max_iter": 200,
            "learning_rate": 0.08,
            "max_leaf_nodes": 31,
            "l2_regularization": 1.0
        },
        "HGB_shallow": {
            "max_iter": 200,
            "learning_rate": 0.08,
            "max_leaf_nodes": 15,
            "l2_regularization": 1.0
        },
        "HGB_more_iterations": {
            "max_iter": 300,
            "learning_rate": 0.05,
            "max_leaf_nodes": 31,
            "l2_regularization": 1.0
        },
        "HGB_regularized": {
            "max_iter": 200,
            "learning_rate": 0.08,
            "max_leaf_nodes": 31,
            "l2_regularization": 5.0
        }
    }
}

tuning_results = []
tuned_models = {}

for model_family, configurations in tuning_configurations.items():

    for config_name, parameters in configurations.items():

        if model_family == "Random Forest":
            model = RandomForestRegressor(
                **parameters,
                n_jobs=-1,
                random_state=42
            )
        else:
            model = HistGradientBoostingRegressor(
                **parameters,
                random_state=42
            )

        print(f"\nTraining {config_name}...", flush=True)

        start_time = time.perf_counter()
        model.fit(X_internal_train, y_internal_train)
        training_seconds = time.perf_counter() - start_time

        prediction = model.predict(X_internal_validation)

        mae = mean_absolute_error(
            y_internal_validation,
            prediction
        )

        rmse = np.sqrt(
            mean_squared_error(y_internal_validation, prediction)
        )

        tuned_models[config_name] = model

        tuning_results.append({
            "model_family": model_family,
            "configuration": config_name,
            "MAE": mae,
            "RMSE": rmse,
            "training_seconds": training_seconds
        })

        print(
            f"MAE: {mae:.6f} | "
            f"RMSE: {rmse:.6f} | "
            f"Training: {training_seconds:.2f}s",
            flush=True
        )

tuning_summary = (
    pd.DataFrame(tuning_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("\nJOINT TUNING RESULTS")
print(tuning_summary.to_string(index=False))


Training RF_baseline...
MAE: 0.822759 | RMSE: 1.272598 | Training: 406.97s

Training RF_shallower...
MAE: 0.828266 | RMSE: 1.278992 | Training: 283.91s

Training RF_more_regularized...
MAE: 0.821732 | RMSE: 1.271697 | Training: 395.45s

Training RF_more_trees...
MAE: 0.822520 | RMSE: 1.272141 | Training: 706.10s

Training HGB_baseline...
MAE: 0.827048 | RMSE: 1.274314 | Training: 10.90s

Training HGB_shallow...
MAE: 0.829917 | RMSE: 1.277237 | Training: 7.81s

Training HGB_more_iterations...
MAE: 0.826320 | RMSE: 1.272541 | Training: 15.60s

Training HGB_regularized...
MAE: 0.826446 | RMSE: 1.273411 | Training: 10.56s

JOINT TUNING RESULTS
        model_family       configuration      MAE     RMSE  training_seconds
       Random Forest RF_more_regularized 0.821732 1.271697        395.446098
       Random Forest       RF_more_trees 0.822520 1.272141        706.097739
       Random Forest         RF_baseline 0.822759 1.272598        406.969700
HistGradientBoosting HGB_more_iterations 0.

In [33]:
# WI-06: Save joint Random Forest and HistGradientBoosting tuning results

tuning_results_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_joint_tuning_results.csv"
)

tuning_summary.to_csv(
    tuning_results_path,
    index=False
)

print("JOINT TUNING RESULTS SAVED")
print("File:", tuning_results_path)
print("Configurations evaluated:", len(tuning_summary))

print("\nBEST CONFIGURATION BY MODEL FAMILY")

best_configurations = (
    tuning_summary
    .sort_values("MAE")
    .groupby("model_family")
    .first()
    .reset_index()
)

print(
    best_configurations[
        ["model_family", "configuration", "MAE", "RMSE", "training_seconds"]
    ].to_string(index=False)
)

JOINT TUNING RESULTS SAVED
File: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_joint_tuning_results.csv
Configurations evaluated: 8

BEST CONFIGURATION BY MODEL FAMILY
        model_family       configuration      MAE     RMSE  training_seconds
HistGradientBoosting HGB_more_iterations 0.826320 1.272541         15.603631
       Random Forest RF_more_regularized 0.821732 1.271697        395.446098


In [34]:
# WI-06: Refit the selected Random Forest configuration
# Full training period: 2013–2019

import time
from sklearn.ensemble import RandomForestRegressor

rf_finalist = RandomForestRegressor(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=10,
    max_features=0.8,
    n_jobs=-1,
    random_state=42
)

print("Training selected Random Forest configuration...", flush=True)

start_time = time.perf_counter()

rf_finalist.fit(X_train, y_train)

rf_finalist_training_seconds = time.perf_counter() - start_time

print("\nRANDOM FOREST FINALIST TRAINED")
print("Training rows:", len(X_train))
print("Training years: 2013–2019")
print(f"Training time: {rf_finalist_training_seconds:.2f} seconds")

Training selected Random Forest configuration...

RANDOM FOREST FINALIST TRAINED
Training rows: 352800
Training years: 2013–2019
Training time: 557.47 seconds


In [35]:
# WI-06: Evaluate the selected Random Forest on 2020–2022

start_time = time.perf_counter()

rf_finalist_prediction = rf_finalist.predict(X_validation)

rf_finalist_prediction_seconds = time.perf_counter() - start_time

rf_finalist_mae = mean_absolute_error(
    y_validation,
    rf_finalist_prediction
)

rf_finalist_rmse = np.sqrt(
    mean_squared_error(y_validation, rf_finalist_prediction)
)

print("RANDOM FOREST FINALIST — VALIDATION PERFORMANCE")
print(f"MAE: {rf_finalist_mae:.6f} °C")
print(f"RMSE: {rf_finalist_rmse:.6f} °C")
print(f"Training time: {rf_finalist_training_seconds:.2f} seconds")
print(f"Prediction time: {rf_finalist_prediction_seconds:.2f} seconds")

print("\nBASELINE COMPARISON")
print(f"Persistence MAE: {baseline_results.loc[0, 'MAE']:.6f} °C")
print(f"Monthly climatology MAE: {baseline_results.loc[1, 'MAE']:.6f} °C")

RANDOM FOREST FINALIST — VALIDATION PERFORMANCE
MAE: 0.813160 °C
RMSE: 1.248906 °C
Training time: 557.47 seconds
Prediction time: 0.84 seconds

BASELINE COMPARISON
Persistence MAE: 0.883598 °C
Monthly climatology MAE: 1.637673 °C


In [36]:
# WI-06: Refit the selected HistGradientBoosting configuration
# Full training period: 2013–2019

from sklearn.ensemble import HistGradientBoostingRegressor
import time

hgb_finalist = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

print("Training selected HistGradientBoosting configuration...", flush=True)

start_time = time.perf_counter()

hgb_finalist.fit(X_train, y_train)

hgb_finalist_training_seconds = time.perf_counter() - start_time

print("\nHISTGRADIENTBOOSTING FINALIST TRAINED")
print("Training rows:", len(X_train))
print("Training years: 2013–2019")
print(f"Training time: {hgb_finalist_training_seconds:.2f} seconds")

Training selected HistGradientBoosting configuration...

HISTGRADIENTBOOSTING FINALIST TRAINED
Training rows: 352800
Training years: 2013–2019
Training time: 11.31 seconds


In [37]:
# WI-06: Evaluate the selected HistGradientBoosting model

start_time = time.perf_counter()

hgb_finalist_prediction = hgb_finalist.predict(X_validation)

hgb_finalist_prediction_seconds = time.perf_counter() - start_time

hgb_finalist_mae = mean_absolute_error(
    y_validation,
    hgb_finalist_prediction
)

hgb_finalist_rmse = np.sqrt(
    mean_squared_error(y_validation, hgb_finalist_prediction)
)

print("HISTGRADIENTBOOSTING FINALIST — VALIDATION PERFORMANCE")
print(f"MAE: {hgb_finalist_mae:.6f} °C")
print(f"RMSE: {hgb_finalist_rmse:.6f} °C")
print(f"Training time: {hgb_finalist_training_seconds:.2f} seconds")
print(f"Prediction time: {hgb_finalist_prediction_seconds:.2f} seconds")

print("\nBASELINE COMPARISON")
print(f"Persistence MAE: {baseline_results.loc[0, 'MAE']:.6f} °C")
print(f"Monthly climatology MAE: {baseline_results.loc[1, 'MAE']:.6f} °C")

HISTGRADIENTBOOSTING FINALIST — VALIDATION PERFORMANCE
MAE: 0.816725 °C
RMSE: 1.247800 °C
Training time: 11.31 seconds
Prediction time: 1.05 seconds

BASELINE COMPARISON
Persistence MAE: 0.883598 °C
Monthly climatology MAE: 1.637673 °C


In [38]:
# WI-06: Compare the refitted finalist models

finalist_comparison = pd.DataFrame([
    {
        "model": "Random Forest",
        "MAE": rf_finalist_mae,
        "RMSE": rf_finalist_rmse,
        "training_seconds": rf_finalist_training_seconds,
        "prediction_seconds": rf_finalist_prediction_seconds
    },
    {
        "model": "HistGradientBoosting",
        "MAE": hgb_finalist_mae,
        "RMSE": hgb_finalist_rmse,
        "training_seconds": hgb_finalist_training_seconds,
        "prediction_seconds": hgb_finalist_prediction_seconds
    },
    {
        "model": "Persistence",
        "MAE": baseline_results.loc[0, "MAE"],
        "RMSE": baseline_results.loc[0, "RMSE"],
        "training_seconds": np.nan,
        "prediction_seconds": np.nan
    },
    {
        "model": "Monthly climatology",
        "MAE": baseline_results.loc[1, "MAE"],
        "RMSE": baseline_results.loc[1, "RMSE"],
        "training_seconds": np.nan,
        "prediction_seconds": np.nan
    }
])

finalist_comparison["MAE_improvement_vs_persistence_pct"] = (
    (baseline_results.loc[0, "MAE"] - finalist_comparison["MAE"])
    / baseline_results.loc[0, "MAE"] * 100
)

comparison_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_finalist_comparison.csv"
)

finalist_comparison.to_csv(comparison_path, index=False)

print("FINALIST COMPARISON")
print(finalist_comparison.to_string(index=False))
print("\nSaved:", comparison_path)

FINALIST COMPARISON
               model      MAE     RMSE  training_seconds  prediction_seconds  MAE_improvement_vs_persistence_pct
       Random Forest 0.813160 1.248906        557.471011            0.838264                            7.971760
HistGradientBoosting 0.816725 1.247800         11.313522            1.049884                            7.568295
         Persistence 0.883598 1.382034               NaN                 NaN                            0.000000
 Monthly climatology 1.637673 2.440288               NaN                 NaN                          -85.341457

Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_finalist_comparison.csv


In [39]:
# WI-06: Analyse prediction bias for the two finalist models

bias_results = []

predictions_to_check = {
    "Random Forest": rf_finalist_prediction,
    "HistGradientBoosting": hgb_finalist_prediction,
    "Persistence": validation_sample["persistence_prediction"].to_numpy()
}

actual_values = y_validation.to_numpy()

for model_name, predictions in predictions_to_check.items():

    errors = predictions - actual_values

    bias_results.append({
        "model": model_name,
        "mean_actual": actual_values.mean(),
        "mean_predicted": predictions.mean(),
        "mean_error": errors.mean(),
        "MAE": mean_absolute_error(actual_values, predictions),
        "RMSE": np.sqrt(mean_squared_error(actual_values, predictions))
    })

bias_summary = (
    pd.DataFrame(bias_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

print("T2M_MAX T+1 PREDICTION BIAS ANALYSIS")
print(bias_summary.to_string(index=False))

print("\nInterpretation:")
print("Positive mean error = systematic overprediction")
print("Negative mean error = systematic underprediction")

T2M_MAX T+1 PREDICTION BIAS ANALYSIS
               model  mean_actual  mean_predicted  mean_error      MAE     RMSE
       Random Forest    25.617662       25.601486   -0.016176 0.813160 1.248906
HistGradientBoosting    25.617662       25.602341   -0.015321 0.816725 1.247800
         Persistence    25.617662       25.618237    0.000575 0.883598 1.382034

Interpretation:
Positive mean error = systematic overprediction
Negative mean error = systematic underprediction


In [40]:
# WI-06: Save prediction bias diagnostics

bias_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_prediction_bias.csv"
)

bias_summary.to_csv(bias_path, index=False)

print("PREDICTION BIAS REPORT SAVED")
print("File:", bias_path)
print("Models evaluated:", len(bias_summary))

PREDICTION BIAS REPORT SAVED
File: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_prediction_bias.csv
Models evaluated: 3


In [41]:
# WI-06: Evaluate errors across actual maximum-temperature ranges

temperature_range_frame = pd.DataFrame({
    "actual": y_validation.to_numpy(),
    "Random_Forest": rf_finalist_prediction,
    "HistGradientBoosting": hgb_finalist_prediction
})

temperature_bins = [-np.inf, 0, 10, 20, 30, 40, np.inf]
temperature_labels = [
    "Below 0°C",
    "0–10°C",
    "10–20°C",
    "20–30°C",
    "30–40°C",
    "Above 40°C"
]

temperature_range_frame["temperature_range"] = pd.cut(
    temperature_range_frame["actual"],
    bins=temperature_bins,
    labels=temperature_labels,
    right=False
)

range_results = []

for temperature_range, group in temperature_range_frame.groupby(
    "temperature_range",
    observed=False
):
    if len(group) == 0:
        continue

    actual = group["actual"].to_numpy()

    for model_name, prediction_col in [
        ("Random Forest", "Random_Forest"),
        ("HistGradientBoosting", "HistGradientBoosting")
    ]:
        predicted = group[prediction_col].to_numpy()
        errors = predicted - actual

        range_results.append({
            "temperature_range": str(temperature_range),
            "model": model_name,
            "rows": len(group),
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "mean_error": errors.mean()
        })

temperature_range_performance = pd.DataFrame(range_results)

print("ERROR BY ACTUAL TEMPERATURE RANGE")
print(temperature_range_performance.to_string(index=False))

ERROR BY ACTUAL TEMPERATURE RANGE
temperature_range                model  rows      MAE     RMSE  mean_error
        Below 0°C        Random Forest  8577 1.749429 2.241600    0.346846
        Below 0°C HistGradientBoosting  8577 1.761972 2.255143    0.316032
           0–10°C        Random Forest  7766 1.502850 1.941507   -0.074640
           0–10°C HistGradientBoosting  7766 1.510517 1.943434   -0.111222
          10–20°C        Random Forest 11207 1.239828 1.664795   -0.164653
          10–20°C HistGradientBoosting 11207 1.237949 1.645340   -0.151684
          20–30°C        Random Forest 81506 0.524196 0.874452    0.081806
          20–30°C HistGradientBoosting 81506 0.530207 0.873576    0.082699
          30–40°C        Random Forest 35797 0.936122 1.293291   -0.181471
          30–40°C HistGradientBoosting 35797 0.937827 1.297212   -0.171300
       Above 40°C        Random Forest  6347 0.967947 1.268073   -0.499026
       Above 40°C HistGradientBoosting  6347 0.943040 1.235654   -

In [42]:
# WI-06: Save temperature-range performance diagnostics

range_performance_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T1_temperature_range_performance.csv"
)

temperature_range_performance.to_csv(
    range_performance_path,
    index=False
)

print("TEMPERATURE-RANGE REPORT SAVED")
print("File:", range_performance_path)
print("Records:", len(temperature_range_performance))

print("\nOBSERVATIONS PER TEMPERATURE RANGE")
print(
    temperature_range_frame["temperature_range"]
    .value_counts(sort=False)
    .to_string()
)

TEMPERATURE-RANGE REPORT SAVED
File: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_temperature_range_performance.csv
Records: 12

OBSERVATIONS PER TEMPERATURE RANGE
temperature_range
Below 0°C      8577
0–10°C         7766
10–20°C       11207
20–30°C       81506
30–40°C       35797
Above 40°C     6347


In [43]:
# WI-06: Compare refitted finalists across Indian meteorological seasons

seasonal_finalist_frame = pd.DataFrame({
    "target_date": pd.to_datetime(validation_sample["target_date"]),
    "actual": y_validation.to_numpy(),
    "Random_Forest": rf_finalist_prediction,
    "HistGradientBoosting": hgb_finalist_prediction
})

seasonal_finalist_frame["month"] = (
    seasonal_finalist_frame["target_date"].dt.month
)

def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Pre-monsoon"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-monsoon"

seasonal_finalist_frame["season"] = (
    seasonal_finalist_frame["month"].apply(assign_season)
)

seasonal_finalist_results = []

for season, group in seasonal_finalist_frame.groupby("season"):

    actual = group["actual"].to_numpy()

    for model_name in ["Random_Forest", "HistGradientBoosting"]:

        predicted = group[model_name].to_numpy()

        seasonal_finalist_results.append({
            "season": season,
            "model": model_name,
            "rows": len(group),
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "mean_error": (predicted - actual).mean()
        })

seasonal_finalist_performance = (
    pd.DataFrame(seasonal_finalist_results)
    .sort_values(["season", "MAE"])
    .reset_index(drop=True)
)

print("REFITTED FINALIST SEASONAL PERFORMANCE")
print(seasonal_finalist_performance.to_string(index=False))

REFITTED FINALIST SEASONAL PERFORMANCE
      season                model  rows      MAE     RMSE  mean_error
     Monsoon        Random_Forest 50020 0.784050 1.165954   -0.026226
     Monsoon HistGradientBoosting 50020 0.785876 1.169055   -0.024912
Post-monsoon HistGradientBoosting 24920 0.689686 1.034902    0.028441
Post-monsoon        Random_Forest 24920 0.690433 1.046975    0.042409
 Pre-monsoon HistGradientBoosting 37730 0.896086 1.387150    0.003935
 Pre-monsoon        Random_Forest 37730 0.898200 1.396628   -0.034323
      Winter        Random_Forest 38530 0.847051 1.317946   -0.023248
      Winter HistGradientBoosting 38530 0.861224 1.326212   -0.050029


In [44]:
# WI-06: Save seasonal performance of refitted finalist models

seasonal_finalist_path = (
    wi06_report_dir
    / "WI_06_T2M_MAX_T1_refitted_seasonal_performance.csv"
)

seasonal_finalist_performance.to_csv(
    seasonal_finalist_path,
    index=False
)

print("REFITTED SEASONAL PERFORMANCE SAVED")
print("File:", seasonal_finalist_path)
print("Records:", len(seasonal_finalist_performance))

print("\nBEST MODEL BY SEASON")

best_seasonal_models = (
    seasonal_finalist_performance
    .sort_values("MAE")
    .groupby("season")
    .first()
    .reset_index()
)

print(
    best_seasonal_models[
        ["season", "model", "MAE", "RMSE"]
    ].to_string(index=False)
)

REFITTED SEASONAL PERFORMANCE SAVED
File: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\phase3b_weather_intelligence\reports\weather_intelligence_v2\WI_06_temperature_model_development\WI_06_T2M_MAX_T1_refitted_seasonal_performance.csv
Records: 8

BEST MODEL BY SEASON
      season                model      MAE     RMSE
     Monsoon        Random_Forest 0.784050 1.165954
Post-monsoon HistGradientBoosting 0.689686 1.034902
 Pre-monsoon HistGradientBoosting 0.896086 1.387150
      Winter        Random_Forest 0.847051 1.317946


In [45]:
# WI-06: Prepare T2M_MAX T+2 samples

target_col_t2 = "T2M_MAX_target_t_plus_2"

train_parts_t2 = []
validation_parts_t2 = []

for batch_index, file in enumerate(extended_files):

    columns = ["cell_id", "date"] + feature_cols + [target_col_t2]

    frame = pd.read_parquet(file, columns=columns)
    frame["date"] = pd.to_datetime(frame["date"])

    # Calculate the target date for a two-day forecast
    frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
    frame["target_year"] = frame["target_date"].dt.year

    # Keep training and validation periods separate
    train_frame = frame[
        frame["target_year"].isin(TRAIN_YEARS)
    ].dropna(subset=feature_cols + [target_col_t2])

    validation_frame = frame[
        frame["target_year"].isin(VALIDATION_YEARS)
    ].dropna(subset=feature_cols + [target_col_t2])

    # Sample 20 observations per grid cell and target year
    for (cell_id, year), group in train_frame.groupby(
        ["cell_id", "target_year"]
    ):
        train_parts_t2.append(
            group.sample(
                n=min(20, len(group)),
                random_state=42 + batch_index * 1000 + year
            )
        )

    for (cell_id, year), group in validation_frame.groupby(
        ["cell_id", "target_year"]
    ):
        validation_parts_t2.append(
            group.sample(
                n=min(20, len(group)),
                random_state=42 + batch_index * 1000 + year
            )
        )

    if (batch_index + 1) % 50 == 0:
        print(f"Processed {batch_index + 1}/{len(extended_files)} batches")

train_sample_t2 = pd.concat(train_parts_t2, ignore_index=True)
validation_sample_t2 = pd.concat(validation_parts_t2, ignore_index=True)

# Save the samples separately
train_sample_t2_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T2_training_sample.parquet"
)

validation_sample_t2_path = (
    wi06_report_dir / "WI_06_T2M_MAX_T2_validation_sample.parquet"
)

train_sample_t2.to_parquet(train_sample_t2_path, index=False)
validation_sample_t2.to_parquet(validation_sample_t2_path, index=False)

print("\nT2M_MAX T+2 SAMPLES CREATED")
print("Training rows:", len(train_sample_t2))
print("Validation rows:", len(validation_sample_t2))

print("\nTraining target years:")
print(train_sample_t2["target_year"].value_counts().sort_index())

print("\nValidation target years:")
print(validation_sample_t2["target_year"].value_counts().sort_index())

print("\nMissing training values:",
      train_sample_t2[feature_cols + [target_col_t2]].isna().sum().sum())

print("Missing validation values:",
      validation_sample_t2[feature_cols + [target_col_t2]].isna().sum().sum())

C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 50/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 100/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 150/252 batches


... (2 outputs left out)

C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame

Processed 250/252 batches


C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)
C:\Users\gsr58\AppData\Local\Temp\ipykernel_15700\1013197846.py:16: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  frame["target_date"] = frame["date"] + pd.Timedelta(days=2)



T2M_MAX T+2 SAMPLES CREATED
Training rows: 352800
Validation rows: 151200

Training target years:
target_year
2013    50400
2014    50400
2015    50400
2016    50400
2017    50400
2018    50400
2019    50400
Name: count, dtype: int64

Validation target years:
target_year
2020    50400
2021    50400
2022    50400
Name: count, dtype: int64

Missing training values: 0
Missing validation values: 0


In [46]:
# WI-06: Calculate T2M_MAX T+2 baseline performance

X_train_t2 = train_sample_t2[feature_cols]
y_train_t2 = train_sample_t2[target_col_t2]

X_validation_t2 = validation_sample_t2[feature_cols]
y_validation_t2 = validation_sample_t2[target_col_t2]

# Persistence forecast: today's maximum temperature
validation_sample_t2["persistence_prediction"] = (
    validation_sample_t2["T2M_MAX_current"]
)

# Calculate target month for climatology
train_sample_t2["target_month"] = (
    train_sample_t2["target_date"].dt.month
)

validation_sample_t2["target_month"] = (
    validation_sample_t2["target_date"].dt.month
)

# Build climatology using training data only
climatology_t2 = (
    train_sample_t2
    .groupby(["cell_id", "target_month"])[target_col_t2]
    .mean()
    .rename("climatology_prediction")
    .reset_index()
)

validation_sample_t2 = validation_sample_t2.merge(
    climatology_t2,
    on=["cell_id", "target_month"],
    how="left",
    validate="many_to_one"
)

# Evaluate both baselines
baseline_results_t2 = []

for name, prediction_col in [
    ("Persistence", "persistence_prediction"),
    ("Monthly climatology", "climatology_prediction")
]:
    actual = y_validation_t2
    predicted = validation_sample_t2[prediction_col]

    baseline_results_t2.append({
        "baseline": name,
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted))
    })

baseline_results_t2 = pd.DataFrame(baseline_results_t2)

print("T2M_MAX T+2 VALIDATION BASELINES")
print(baseline_results_t2.to_string(index=False))

print("\nMissing climatology predictions:",
      validation_sample_t2["climatology_prediction"].isna().sum())

T2M_MAX T+2 VALIDATION BASELINES
           baseline      MAE     RMSE
        Persistence 1.183139 1.808661
Monthly climatology 1.640393 2.446088

Missing climatology predictions: 0


In [47]:
# WI-06: Define T2M_MAX T+2 candidate models

candidate_models_t2 = {
    "Ridge": make_pipeline(
        StandardScaler(),
        Ridge(alpha=10.0)
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=20,
        min_samples_leaf=5,
        max_features=0.8,
        n_jobs=-1,
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=150,
        max_depth=20,
        min_samples_leaf=5,
        max_features=0.8,
        n_jobs=-1,
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.08,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    )
}

print("T2M_MAX T+2 CANDIDATE MODELS")

for model_name, model in candidate_models_t2.items():
    print(f"{model_name}: {type(model).__name__}")

print("\nTraining rows:", len(X_train_t2))
print("Validation rows:", len(X_validation_t2))
print("Predictors:", len(feature_cols))

T2M_MAX T+2 CANDIDATE MODELS
Ridge: Pipeline
Random Forest: RandomForestRegressor
Extra Trees: ExtraTreesRegressor
HistGradientBoosting: HistGradientBoostingRegressor

Training rows: 352800
Validation rows: 151200
Predictors: 53


In [48]:
# WI-06: Train and evaluate Ridge Regression for T2M_MAX T+2

import time

ridge_t2 = candidate_models_t2["Ridge"]

start_time = time.perf_counter()
ridge_t2.fit(X_train_t2, y_train_t2)
ridge_t2_training_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()
ridge_t2_prediction = ridge_t2.predict(X_validation_t2)
ridge_t2_prediction_seconds = time.perf_counter() - start_time

ridge_t2_mae = mean_absolute_error(
    y_validation_t2, ridge_t2_prediction
)

ridge_t2_rmse = np.sqrt(
    mean_squared_error(y_validation_t2, ridge_t2_prediction)
)

benchmark_results_t2 = [{
    "model": "Ridge",
    "MAE": ridge_t2_mae,
    "RMSE": ridge_t2_rmse,
    "training_seconds": ridge_t2_training_seconds,
    "prediction_seconds": ridge_t2_prediction_seconds
}]

print("RIDGE REGRESSION — T2M_MAX T+2")
print(f"MAE: {ridge_t2_mae:.6f} °C")
print(f"RMSE: {ridge_t2_rmse:.6f} °C")
print(f"Training time: {ridge_t2_training_seconds:.2f} seconds")
print(f"Prediction time: {ridge_t2_prediction_seconds:.2f} seconds")
print(f"Persistence MAE: {baseline_results_t2.loc[0, 'MAE']:.6f} °C")

RIDGE REGRESSION — T2M_MAX T+2
MAE: 1.103405 °C
RMSE: 1.656611 °C
Training time: 0.93 seconds
Prediction time: 0.09 seconds
Persistence MAE: 1.183139 °C


In [49]:
# WI-06: Train and evaluate Random Forest for T2M_MAX T+2

import time

rf_t2 = candidate_models_t2["Random Forest"]

start_time = time.perf_counter()
rf_t2.fit(X_train_t2, y_train_t2)
rf_t2_training_seconds = time.perf_counter() - start_time

start_time = time.perf_counter()
rf_t2_prediction = rf_t2.predict(X_validation_t2)
rf_t2_prediction_seconds = time.perf_counter() - start_time

rf_t2_mae = mean_absolute_error(y_validation_t2, rf_t2_prediction)
rf_t2_rmse = np.sqrt(
    mean_squared_error(y_validation_t2, rf_t2_prediction)
)

benchmark_results_t2.append({
    "model": "Random Forest",
    "MAE": rf_t2_mae,
    "RMSE": rf_t2_rmse,
    "training_seconds": rf_t2_training_seconds,
    "prediction_seconds": rf_t2_prediction_seconds
})

print("RANDOM FOREST — T2M_MAX T+2")
print(f"MAE: {rf_t2_mae:.6f} °C")
print(f"RMSE: {rf_t2_rmse:.6f} °C")
print(f"Training time: {rf_t2_training_seconds:.2f} seconds")
print(f"Prediction time: {rf_t2_prediction_seconds:.2f} seconds")

print("\nCURRENT BENCHMARK")
print(pd.DataFrame(benchmark_results_t2).to_string(index=False))

RANDOM FOREST — T2M_MAX T+2
MAE: 1.051751 °C
RMSE: 1.593270 °C
Training time: 593.61 seconds
Prediction time: 1.21 seconds

CURRENT BENCHMARK
        model      MAE     RMSE  training_seconds  prediction_seconds
        Ridge 1.103405 1.656611          0.925452            0.090374
Random Forest 1.051751 1.593270        593.613777            1.208365
